KONTROL LANGKAH untuk learning curve v2 (protokol dari diskusi `2026-11-23-menambah-data-sesudah-learning-curve`).
Pertanyaan: apakah keunggulan f100 atas f050 bertahan bila jumlah langkah terjadwal disamakan? Hanya VALID; test tidak dibaca.

- Lengan: f100 (ulang), f050_s0, f050_s1 (grup identik dengan v2, dicek dengan `grup_sha256`).
- Semua lengan: total langkah terjadwal = 1.905 (= 15 epoch x 127 batch f100), warmup 10 % = 190, jadwal linear sama.
  f050 (64 batch/epoch) berhenti di 29 epoch + 49 batch.
- Angka utama: rerank pada bobot TITIK AKHIR. Sekunder: checkpoint pilihan aturan demo (chrF++ + 30 x juri kernel).
- Update optimizer aktual dicatat (langkah yang dilewati GradScaler karena inf/nan).
- Cek kewarasan: greedy chrF++ per epoch f100 harus identik dengan f100 di learning_curve_v2.
- Aturan baca (dikunci sebelum push, [asumsi], bukan uji signifikansi), atas gap G = chrF++ total titik akhir f100 - f050_sX:
  kedua G >= 3,5 -> "keunggulan f100 bertahan pada anggaran langkah yang sama"; kedua |G| <= 1,0 -> "kedua subset mendekati
  f100 pada kontrol ini"; kedua G < -1,0 -> pembalikan; selain itu -> tidak ada klaim mekanisme.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu", "sentencepiece"], check=False)

import glob, json, time, random, re, os, hashlib, gc, shutil
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np, pandas as pd, torch, sacrebleu, transformers
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

REPO, SL, TL = "facebook/m2m100_418M", "id", "jv"
TRAIN_SEED = 42
EPOCHS, OS, BS, LR, WARMUP, JUDGE_SEED, MAXLEN, CTRL_W = 15, 3, 16, 5e-5, 0.1, 42, 64, 30.0
NBEST, LAM = 8, 2.0
OUT = "/kaggle/working"
dev = "cuda"
LEVELS = ["ngoko", "krama", "krama_alus"]
LID = {l: i for i, l in enumerate(LEVELS)}
TAG = {l: f"<{l}>" for l in LEVELS}
INIT_WORDS = {"ngoko": "kowe aku mangan lunga turu", "krama": "sampeyan kula nedha kesah tilem",
              "krama_alus": "panjenenganipun dalem dhahar tindak sare"}
T0 = time.time()
log = lambda *a: print(f"[{time.time()-T0:6.0f}s]", *a, flush=True)
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


find = lambda n: sorted(p for p in glob.glob(f"/kaggle/input/**/{n}.csv", recursive=True) if "/output" not in p)[0]
tr_raw = pd.read_csv(find("train"))
def _load3(n):
    d = pd.read_csv(find(n))
    d = d[d["level"].isin(LEVELS)].copy()
    d["label"] = d["level"].map(LID)
    return d.reset_index(drop=True)
tr, va, te = (_load3(n) for n in ["train", "valid", "test"])
assert (len(tr), len(va), len(te)) == (1395, 175, 176), (len(tr), len(va), len(te))
RR_DIR = os.path.dirname(glob.glob("/kaggle/input/**/model/reranker/config.json", recursive=True)[0])
log("data", len(tr), len(va), len(te), torch.cuda.get_device_name(0), "| reranker", RR_DIR)
tag_src = lambda ind, level: f"{TAG[level]} {ind}"
def bleu(h, r): return sacrebleu.corpus_bleu(h, [r]).score
def chrf(h, r): return sacrebleu.corpus_chrf(h, [r], word_order=2).score
_rr_w = glob.glob(f"{RR_DIR}/*.safetensors") + glob.glob(f"{RR_DIR}/*.bin")
MANIFEST = {"protokol": "kontrol langkah (diskusi 2026-11-23-menambah-data-sesudah-learning-curve)", "train_seed": TRAIN_SEED,
            "hparams": {"epochs": EPOCHS, "os": OS, "bs": BS, "lr": LR, "warmup": WARMUP, "maxlen": MAXLEN,
                        "ctrl_w": CTRL_W, "nbest": NBEST, "lambda": LAM, "judge_seed": JUDGE_SEED},
            "versions": {"python": sys.version.split()[0], "torch": torch.__version__,
                         "transformers": transformers.__version__, "sacrebleu": sacrebleu.__version__},
            "gpu": torch.cuda.get_device_name(0),
            "inputs": {n: sha(find(n)) for n in ["train", "valid", "test"]},
            "reranker_sha256": sha(_rr_w[0]) if _rr_w else None,
            "runs": {}}

In [ ]:
# ---------------------------------------------------- juri kernel (kode v2 persis) + reranker demo (dimuat)
def train_classifier(base, seed):
    ctok = AutoTokenizer.from_pretrained(base)
    cmod = AutoModelForSequenceClassification.from_pretrained(base, num_labels=len(LEVELS)).to(dev)
    seed_all(seed)
    opt = torch.optim.AdamW(cmod.parameters(), lr=5e-5)
    X, y = tr["javanese"].tolist(), tr["label"].tolist()
    for _ in range(4):
        cmod.train(); order = list(range(len(X))); random.shuffle(order)
        for i in range(0, len(order), 32):
            b = order[i:i + 32]
            enc = ctok([X[j] for j in b], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
            loss = cmod(**enc, labels=torch.tensor([y[j] for j in b], device=dev)).loss
            opt.zero_grad(); loss.backward(); opt.step()
    cmod.eval()
    return ctok, cmod


def make_lp(ctok, cmod):
    def logprobs(sents):
        out = []
        for i in range(0, len(sents), 64):
            enc = ctok(sents[i:i + 64], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
            with torch.no_grad():
                out.append(F.log_softmax(cmod(**enc).logits.float(), -1).cpu())
        return torch.cat(out).numpy()
    return logprobs


def clf_info(lp):
    info = {}
    for n, d in [("valid", va), ("test", te)]:
        p = lp(d["javanese"].tolist()).argmax(-1); y_ = d["label"].values
        info[f"acc_{n}"] = round(float((p == y_).mean()), 4)
        info[f"per_class_acc_{n}"] = {l: round(float((p[y_ == i] == i).mean()), 4) for i, l in enumerate(LEVELS)}
    return info


judge_lp = make_lp(*train_classifier("w11wo/javanese-distilbert-small", JUDGE_SEED))
classify = lambda s: judge_lp(s).argmax(-1).tolist()
rr_tok = AutoTokenizer.from_pretrained(RR_DIR)
rr_mod = AutoModelForSequenceClassification.from_pretrained(RR_DIR).to(dev).float().eval()
rr_lp = make_lp(rr_tok, rr_mod)
# v2: juri acc_valid 0.96 (KA 0.8148), test 0.9375; reranker acc_valid 0.9657, test 0.9261
MANIFEST["judge"] = clf_info(judge_lp); MANIFEST["reranker"] = clf_info(rr_lp)
log("juri", MANIFEST["judge"]); log("reranker", MANIFEST["reranker"])

In [ ]:
# ---------------------------------------------------- MT (fungsi v2 persis)
def load():
    tok = AutoTokenizer.from_pretrained(REPO, src_lang=SL, tgt_lang=TL)
    model = AutoModelForSeq2SeqLM.from_pretrained(REPO)
    tok.add_tokens(list(TAG.values()), special_tokens=True)
    model.resize_token_embeddings(len(tok))
    emb = model.get_input_embeddings().weight
    with torch.no_grad():
        for l, words in INIT_WORDS.items():
            ids = tok(words, add_special_tokens=False)["input_ids"]
            emb[tok.convert_tokens_to_ids(TAG[l])] = emb[ids].mean(0)
    bos = tok.get_lang_id(TL)
    return tok, model.to(dev), bos


def translate(tok, model, bos, srcs, beams=4, bs=48):
    model.eval(); out = [None] * len(srcs)
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([srcs[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            g = model.generate(**enc, forced_bos_token_id=bos, num_beams=beams, max_new_tokens=MAXLEN)
        for j, t in zip(idx, tok.batch_decode(g, skip_special_tokens=True)):
            out[j] = t
    return out


def seq_scores(tok, model, srcs, cands, chunk=16):
    out = []
    for i in range(0, len(srcs), chunk):
        enc = tok(srcs[i:i + chunk], text_target=cands[i:i + chunk], return_tensors="pt",
                  padding=True, truncation=True, max_length=MAXLEN).to(dev)
        labels = enc.pop("labels"); labels[labels == tok.pad_token_id] = -100
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc, labels=labels).logits
        nll = F.cross_entropy(logits.float().transpose(1, 2), labels, ignore_index=-100, reduction="none")
        m = (labels != -100).float()
        out += (-(nll * m).sum(1) / m.sum(1).clamp(min=1)).cpu().tolist()
        del logits, nll
    return np.array(out)


def nbest(tok, model, bos, srcs, k=NBEST, bs=8):
    model.eval(); out = [None] * len(srcs)
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([srcs[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            g = model.generate(**enc, forced_bos_token_id=bos, num_beams=k, num_return_sequences=k, max_new_tokens=MAXLEN)
        txt = tok.batch_decode(g, skip_special_tokens=True)
        sc = seq_scores(tok, model, [srcs[j] for j in idx for _ in range(k)], txt)
        for n, j in enumerate(idx):
            out[j] = (txt[n * k:(n + 1) * k], sc[n * k:(n + 1) * k])
    torch.cuda.empty_cache()
    return out


def rerank(nb, levels, lam=LAM):
    flat = [t for c, _ in nb for t in c]
    lp = rr_lp(flat); res = []; p = 0
    for (c, s), l in zip(nb, levels):
        k = len(c); tot = s + lam * lp[p:p + k, LID[l]]; p += k
        res.append(c[int(np.argmax(tot))])
    return res


def ctrl_acc(hyps, levels):
    p = np.array(classify(hyps)); y = np.array([LID[l] for l in levels])
    return float((p == y).mean()), {l: round(float((p[y == i] == i).mean()), 4) for i, l in enumerate(LEVELS)}


def valid_score(h):
    c = chrf(h, va["javanese"].tolist()); a, alv = ctrl_acc(h, va["level"].tolist())
    return c + CTRL_W * a, round(c, 2), round(a, 4), alv


def per_level(h, d):
    res = {}
    for lv in LEVELS + ["all"]:
        m = (d["level"] == lv) if lv != "all" else d["level"].notna()
        hh, r = [x for x, k in zip(h, m) if k], d.loc[m, "javanese"].tolist()
        res[lv] = {"n": len(r), "chrF++": round(chrf(hh, r), 2), "BLEU": round(bleu(hh, r), 2)}
    return res


def honor_pairs(raw):
    pairs = []
    for _, g in raw.groupby("group"):
        jv = dict(zip(g["level"], g["javanese"]))
        if "krama" in jv and "krama_alus" in jv and jv["krama"] != jv["krama_alus"]:
            pairs.append((f"{TAG['krama_alus']} {jv['krama']}", jv["krama_alus"]))
    return pairs

In [ ]:
# ---------------------------------------------------- FASE VALID untuk ketiga seed
def train_run(name, tr_sub, raw_sub, total_steps, seed=TRAIN_SEED):
    seed_all(seed)
    tok, model, bos = load()
    base = [(tag_src(s, l), t, l) for s, l, t in zip(tr_sub["indonesian"], tr_sub["level"], tr_sub["javanese"])]
    id_pairs = [(s, t) for s, t, _ in base] + [(s, t) for s, t, l in base if l == "krama_alus"] * (OS - 1)
    jv_pairs = honor_pairs(raw_sub)

    def make_batches():
        a, b = id_pairs[:], jv_pairs[:]
        random.shuffle(a); random.shuffle(b)
        bt = [(SL, a[i:i + BS]) for i in range(0, len(a), BS)] + [(TL, b[i:i + BS]) for i in range(0, len(b), BS)]
        random.shuffle(bt); return bt

    n_batches = len(make_batches())
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = total_steps
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    best, best_state, hist = -1, None, []
    va_src = [tag_src(s, l) for s, l in zip(va["indonesian"], va["level"])]
    step = n_upd = ep = 0
    while step < steps:
        model.train(); tot = n = 0
        for src_lang, b in make_batches():
            if step >= steps:
                break
            tok.src_lang = src_lang
            enc = tok([s for s, _ in b], text_target=[t for _, t in b], return_tensors="pt",
                      padding=True, truncation=True, max_length=MAXLEN).to(dev)
            labels = enc.pop("labels"); labels[labels == tok.pad_token_id] = -100
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**enc, labels=labels).logits
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), labels.view(-1),
                                   ignore_index=-100, label_smoothing=0.0)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            s0 = scaler.get_scale(); scaler.step(opt); scaler.update()
            n_upd += int(scaler.get_scale() >= s0); sched.step(); step += 1
            tot += loss.item() * len(b); n += len(b)
        ep += 1
        tok.src_lang = SL
        sc, c, a, alv = valid_score(translate(tok, model, bos, va_src, 1))
        hist.append({"epoch": ep, "step": step, "train_loss": round(tot / n, 4), "valid_chrF++": c, "valid_ctrl": a,
                     "valid_ctrl_lv": alv, "score": round(sc, 2)})
        log(name, hist[-1])
        if sc > best:
            best = sc; best_state = {k: x.detach().to("cpu", copy=True) for k, x in model.state_dict().items()}
    tok.src_lang = SL
    del opt, scaler, sched; gc.collect(); torch.cuda.empty_cache()
    out = {}
    for tag in ["akhir", "pilih"]:
        if tag == "pilih":
            model.load_state_dict(best_state); del best_state; gc.collect()
        hyp = rerank(nbest(tok, model, bos, va_src), va["level"].tolist())
        sc, c, a, alv = valid_score(hyp)
        pv = va.copy(); pv["hyp"] = hyp; pv["juri_kernel"] = [LEVELS[q] for q in classify(hyp)]
        pv.to_csv(f"{OUT}/valid_{name}_{tag}.csv", index=False)
        out[tag] = {                   "valid_rerank": {"score": round(sc, 2), "chrF++": c, "ctrl": a, "ctrl_lv": alv, "per_level": per_level(hyp, va)},
                   "valid_csv_sha256": sha(f"{OUT}/valid_{name}_{tag}.csv")}
    res = {"n_pairs": [len(id_pairs), len(jv_pairs)], "n_batches_per_epoch": n_batches, "steps_scheduled": steps,
           "steps_done": step, "optimizer_updates": n_upd, "warmup": int(WARMUP * steps), "history": hist,
           "best_epoch": max(hist, key=lambda h: h["score"])["epoch"], **out}
    del model; gc.collect(); torch.cuda.empty_cache()
    log(f"{name} VALID", res["steps_done"], res["optimizer_updates"], res["akhir"]["valid_rerank"]["chrF++"], res["pilih"]["valid_rerank"]["chrF++"])
    return res


# ---------------------------------------------------- lengan kontrol
g_all = sorted(tr["group"].unique())
g_ka = sorted(tr.loc[tr["level"] == "krama_alus", "group"].unique())
g_no = [g for g in g_all if g not in set(g_ka)]
n_ka = int((tr["level"] == "krama_alus").sum())
NB100 = -(-(len(tr) + n_ka * (OS - 1)) // BS) + -(-len(honor_pairs(tr_raw)) // BS)
TOTAL = EPOCHS * NB100
assert (NB100, TOTAL) == (127, 1905), (NB100, TOTAL)
SHA_V2 = {"f050_s0": "b7257006a12edcdf5e206850bc94d9076d4cb216ad43ef65d4ea0c10f5093077",
          "f050_s1": "fddb2f67019e12ad20f195ad22256c424c5453e80493035f4f70123c84e3cb33"}
MANIFEST["total_steps"] = TOTAL
for frac, ss in [(1.0, None), (0.5, 0), (0.5, 1)]:
    if ss is None:
        keep = set(g_all)
    else:
        rng = random.Random(ss)
        pk, pn = g_ka[:], g_no[:]
        rng.shuffle(pk); rng.shuffle(pn)
        keep = set(pk[:round(frac * len(pk))]) | set(pn[:round(frac * len(pn))])
    name = f"f{int(frac * 100):03d}" + ("" if ss is None else f"_s{ss}")
    gsha = hashlib.sha256(",".join(map(str, sorted(keep))).encode()).hexdigest()
    if name in SHA_V2:
        assert gsha == SHA_V2[name], name
    tr_sub = tr[tr["group"].isin(keep)].reset_index(drop=True)
    raw_sub = tr_raw[tr_raw["group"].isin(keep)]
    t = time.time()
    r = train_run(name, tr_sub, raw_sub, TOTAL)
    r.update({"frac": frac, "sub_seed": ss, "n_grup": len(keep), "n_kalimat": len(tr_sub),
              "n_per_level": tr_sub["level"].value_counts().to_dict(), "grup_sha256": gsha, "detik": round(time.time() - t)})
    MANIFEST["runs"][name] = r
    json.dump(MANIFEST, open(f"{OUT}/manifest.json", "w"), indent=1)

V2_F100 = [31.94, 44.32, 51.49, 54.33, 57.35, 57.59, 59.6, 59.83, 60.03, 61.03, 61.1, 61.69, 61.62, 61.73, 61.77]
MANIFEST["cek_f100_vs_lc_v2"] = {"didapat": [h["valid_chrF++"] for h in MANIFEST["runs"]["f100"]["history"]], "harus": V2_F100}
c100 = MANIFEST["runs"]["f100"]["akhir"]["valid_rerank"]["chrF++"]
G = {n: round(c100 - MANIFEST["runs"][n]["akhir"]["valid_rerank"]["chrF++"], 2) for n in ["f050_s0", "f050_s1"]}
g = list(G.values())
if all(x >= 3.5 for x in g): baca = "keunggulan f100 bertahan pada anggaran langkah yang sama"
elif all(abs(x) <= 1.0 for x in g): baca = "kedua subset mendekati f100 pada kontrol ini"
elif all(x < -1.0 for x in g): baca = "pembalikan"
else: baca = "tidak ada klaim mekanisme"
MANIFEST["gap_akhir"] = {**G, "rata": round(sum(g) / 2, 2), "aturan_baca": baca}
MANIFEST["elapsed_s"] = round(time.time() - T0)
json.dump(MANIFEST, open(f"{OUT}/manifest.json", "w"), indent=1)
log("cek f100", MANIFEST["cek_f100_vs_lc_v2"]["didapat"] == V2_F100)
log("GAP", MANIFEST["gap_akhir"])
log("SELESAI")